# 04 — Naive (Metadata-Only) Baseline: EDA and Combination-Strategy Comparison

A real, honest attempt at finding the *best possible* naive baseline for Sonic Explorer —
not just one arbitrary combination rule presented as "the" naive approach. Scope: real EDA
on the four available metadata signals (genre, genre hierarchy, album co-occurrence, tags),
then 3-4 genuinely different combination strategies, evaluated on genre-cohesion@10 — the
exact same metric (`sonic_explorer.evaluation.genre_cohesion`) already used to evaluate every
audio facet elsewhere in this project, so the naive baseline is judged on the same scale.

**Not an exhaustive hyperparameter search.** 3-4 reasoned variants is enough to call this a
genuine attempt, not a strawman — per the project's own discipline (see `analysis/network_graph.py`'s
docstrings): a fair comparison against the real audio-based approach needs the naive baseline to be
the *strongest defensible* version, not whichever combination was quickest to write.

**Where this feeds back:** whichever variant wins becomes `sonic_explorer.analysis.network_graph.DEFAULT_METADATA_WEIGHTS`
— the actual naive baseline shown in the live app (Overview §2, Results §1). This notebook's
conclusion is not academic; it's the literal source of that constant.

## 1. Setup

No GPU, no CLAP, no Demucs — this analysis only touches the `songs` table (genre/album/tag
metadata already recovered by `scripts/enrich_fma_metadata.py`) and some numpy/sklearn math, so
the base package install is enough (no `[colab]` extras needed).

In [ ]:
import os
import subprocess
import sys

REPO_URL = 'https://github.com/oyoai/sonic-explorer.git'
REPO_DIR = '/content/sonic-explorer'


def run(cmd):
    print('$', ' '.join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f'Command failed (exit {result.returncode}): {" ".join(cmd)}')


if os.path.exists(f'{REPO_DIR}/.git'):
    run(['git', '-C', REPO_DIR, 'pull'])
else:
    run(['git', 'clone', REPO_URL, REPO_DIR])

run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO_DIR])

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('sonic_explorer installed from', REPO_DIR)

## 2. Load the real library

Mounts Drive and points at the enriched DB — `genre_top`, `genres_all`, `album_id`, and
`track_tags` must already be populated by `scripts/enrich_fma_metadata.py` (the base ingestion
pipeline strips these at first parse; see that script's docstring). If your Drive copy predates
enrichment, re-run `enrich_fma_metadata.py --db <path>` against it first, or upload a copy of
the local `data/artifacts/sonic_explorer.db` instead (adjust `DB_PATH` below).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/SonicExplorer')
DB_PATH = DRIVE_ROOT / 'artifacts' / 'sonic_explorer.db'

print('DB path:', DB_PATH, '-- exists:', DB_PATH.exists())

In [ ]:
import json

from sonic_explorer.repository.db import init_db
from sonic_explorer.repository.song_repository import SongRepository

conn = init_db(str(DB_PATH))
song_repo = SongRepository(conn)
songs = song_repo.list_songs()
print(f'Loaded {len(songs)} songs')

n_missing_enrichment = sum(1 for s in songs if s.genres_all is None and s.album_id is None and s.track_tags is None)
if n_missing_enrichment == len(songs):
    raise RuntimeError(
        'None of the loaded songs have genres_all/album_id/track_tags populated -- '
        'this DB predates scripts/enrich_fma_metadata.py. Re-run enrichment before continuing.'
    )
print(f'{len(songs) - n_missing_enrichment}/{len(songs)} songs have enrichment data')

## 3. EDA — how sparse/dense is each signal?

Before combining anything: what does each of the four non-audio signals actually look like at
library scale? A signal that's empty for 99% of songs isn't worth the same weight as one that's
reliably populated — this section measures that directly rather than assuming it.

In [ ]:
import numpy as np
import pandas as pd

genre_counts = pd.Series([s.genre_top for s in songs]).value_counts()
print('Genre distribution (8 genres, by design a stratified sample):')
print(genre_counts)
print()

genres_all_sets = [frozenset(json.loads(s.genres_all)) if s.genres_all else frozenset() for s in songs]
n_with_subgenres = sum(1 for s in genres_all_sets if s)
avg_subgenres = np.mean([len(s) for s in genres_all_sets if s]) if n_with_subgenres else 0.0
print(f'genres_all: {n_with_subgenres}/{len(songs)} songs have sub-genre tags '
      f'({n_with_subgenres/len(songs):.1%}), avg {avg_subgenres:.2f} sub-genres per song when present')
print()

album_ids = [s.album_id for s in songs]
n_with_album = sum(1 for a in album_ids if a is not None)
album_counts = pd.Series([a for a in album_ids if a is not None]).value_counts()
n_multi_song_albums = (album_counts > 1).sum()
n_songs_in_multi_song_album = album_counts[album_counts > 1].sum()
print(f'album_id: {n_with_album}/{len(songs)} songs have an album ({n_with_album/len(songs):.1%})')
print(f'  {len(album_counts)} distinct albums, {n_multi_song_albums} shared by >1 curated song')
print(f'  {n_songs_in_multi_song_album}/{len(songs)} songs share an album with another curated song')
print()

tag_sets = [frozenset(json.loads(s.track_tags)) if s.track_tags else frozenset() for s in songs]
n_with_tags = sum(1 for t in tag_sets if t)
avg_tags = np.mean([len(t) for t in tag_sets if t]) if n_with_tags else 0.0
print(f'track_tags: {n_with_tags}/{len(songs)} songs have any tags ({n_with_tags/len(songs):.1%}), '
      f'avg {avg_tags:.2f} tags per song when present')

### Real result (full ~1,400-song local library, run once against `data/artifacts/sonic_explorer.db`)

```
genres_all: 1400/1400 songs have sub-genre tags (100.0%), avg 2.21 sub-genres per song when present
album_id:   1400/1400 songs have an album (100.0%)
            965 distinct albums, 296 shared by >1 curated song
            731/1400 songs share an album with another curated song
track_tags: 221/1400 songs have any tags (15.8%), avg 5.82 tags per song when present
```

`genres_all` and `album_id` are structural FMA fields — reliably populated regardless of
popularity, as expected. `track_tags` came back sparse (15.8% fill rate), also as expected for
FMA's genre-balanced "small" split (uploader-supplied free text, not curated) — it's included in
the combination anyway per an earlier explicit decision, but this is the honest reason it
contributes comparatively little below.

### Cross-signal overlap: does sharing an album mean sharing a genre?

A quick, cheap check worth running before combining anything: if albums are almost always
single-genre, `album_id` mostly just re-derives what `genre_top` already says (redundant, not
complementary). If they diverge meaningfully, `album_id` is contributing real, independent signal.

In [ ]:
same_album_pairs = 0
same_album_same_genre = 0
by_album = {}
for s in songs:
    if s.album_id is not None:
        by_album.setdefault(s.album_id, []).append(s)

for album_songs in by_album.values():
    for i in range(len(album_songs)):
        for j in range(i + 1, len(album_songs)):
            same_album_pairs += 1
            if album_songs[i].genre_top == album_songs[j].genre_top:
                same_album_same_genre += 1

if same_album_pairs:
    print(f'{same_album_pairs} same-album song pairs; '
          f'{same_album_same_genre/same_album_pairs:.1%} of them also share genre_top')

**Real result:** 674 same-album song pairs; **99.3%** (669/674) also share `genre_top`.
`album_id` turns out to be almost entirely redundant with genre in this library — albums are
overwhelmingly single-genre, as you'd expect. The remaining sliver (5 pairs, 0.7%) genuinely
cross a genre boundary — exactly the kind of connection a genre-only rule could never find,
however rare. This tempers expectations for how much *independent* signal `album_id` can
realistically add, ahead of §5's weighting comparison.

## 4. The matching rule for each signal, explicitly

No black box — every similarity score below is one of these four simple, auditable rules:

| Signal | What counts as a match | Formula |
|---|---|---|
| **Genre** (`genre_top`) | Exact string equality of the single top-level genre | `1.0` if equal, else `0.0` |
| **Genre hierarchy** (`genres_all`) | Overlap of FMA's fuller sub-genre-ID sets | Jaccard: `\|A ∩ B\| / \|A ∪ B\|` (`0.0` if both empty — "neither has sub-genres" isn't evidence of similarity) |
| **Album** (`album_id`) | Exact equality of a non-null album ID | `1.0` if equal and both non-null, else `0.0` (two songs both *missing* an album never match each other) |
| **Tags** (`track_tags`) | Overlap of free-text tag-string sets | Jaccard, same convention as genre hierarchy |

This exactly mirrors `sonic_explorer.analysis.network_graph.compute_metadata_similarity_components()`
— the code below calls that function directly rather than reimplementing it, so there is no
drift between what this notebook evaluates and what the app actually runs.

## 5. Combination strategies

Four variants, each combining the same four [0,1] similarity matrices differently:

1. **Equal weight** — the naive starting point: `genre=1, genres_all=1, album=1, tags=1`.
2. **Genre-heavy** — genre is the densest, most reliable signal; tags is the sparsest and noisiest.
   Hypothesis: leaning harder on genre and less on tags should help.
   `genre=2, genres_all=1, album=1, tags=0.5`.
3. **Tags/album-heavy** — the opposite hypothesis: album and tags are *rarer* but, when present,
   more *specific* than genre — up-weighting them might let real cross-genre connections win
   more often. `genre=1, genres_all=1, album=2, tags=2`.
4. **Learned (logistic regression)** — fit a basic logistic regression predicting "same
   `genre_top`" from the four raw signal values on a sample of song pairs, then use the fitted
   model's probability as the combined score. **Honest caveat:** since no independently-labeled
   "these two songs are truly alike" ground truth exists yet in this project (that's what the
   still-empty calibration/XAB ratings are for — see Methodology §8), genre-match is the only
   available supervision signal at this stage, and it's also one of the four input features —
   this is a real, known limitation of this specific variant, not swept under the rug.

In [ ]:
from sonic_explorer.analysis.network_graph import SongMetadata, combine_metadata_similarities, compute_metadata_similarity_components


def song_metadata_from_song(song) -> SongMetadata:
    genres_all = frozenset(json.loads(song.genres_all)) if song.genres_all else frozenset()
    tags = frozenset(json.loads(song.track_tags)) if song.track_tags else frozenset()
    return SongMetadata(genre_top=song.genre_top, genres_all=genres_all, album_id=song.album_id, tags=tags)


song_metadata = {s.id: song_metadata_from_song(s) for s in songs}
genre_by_song = {s.id: s.genre_top for s in songs}

song_ids, components = compute_metadata_similarity_components(song_metadata)
print('Similarity components computed:')
for name, mat in components.items():
    print(f'  {name:12s} mean={mat.mean():.4f}  nonzero_frac={(mat > 0).sum() / mat.size:.4f}')

In [ ]:
variants = {}

variants['equal_weight'] = combine_metadata_similarities(
    components, weights={'genre': 1.0, 'genres_all': 1.0, 'album': 1.0, 'tags': 1.0}
)
variants['genre_heavy'] = combine_metadata_similarities(
    components, weights={'genre': 2.0, 'genres_all': 1.0, 'album': 1.0, 'tags': 0.5}
)
variants['tags_album_heavy'] = combine_metadata_similarities(
    components, weights={'genre': 1.0, 'genres_all': 1.0, 'album': 2.0, 'tags': 2.0}
)

# Learned combination: logistic regression on a random sample of song pairs
# (not all ~980k pairs -- 30k is plenty for 4 features and keeps this fast).
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(42)
n = len(song_ids)
n_train_pairs = 30000
idx_i = rng.integers(0, n, size=n_train_pairs)
idx_j = rng.integers(0, n, size=n_train_pairs)
mask = idx_i != idx_j
idx_i, idx_j = idx_i[mask], idx_j[mask]

feature_names = ['genre', 'genres_all', 'album', 'tags']
X_train = np.stack([components[name][idx_i, idx_j] for name in feature_names], axis=1)
y_train = components['genre'][idx_i, idx_j]  # same genre_top -- see the honest caveat above

clf = LogisticRegression()
clf.fit(X_train, y_train)
print(f'Logistic regression fit on {len(idx_i)} sampled pairs:')
for name, coef in zip(feature_names, clf.coef_[0]):
    print(f'  coef[{name}] = {coef:.4f}')
print(f'  intercept = {clf.intercept_[0]:.4f}')

stacked = np.stack([components[name] for name in feature_names], axis=-1)
logits = stacked @ clf.coef_[0] + clf.intercept_[0]
variants['learned_logistic'] = 1.0 / (1.0 + np.exp(-logits))

### Real result

```
Similarity components computed:
  genre        mean=0.1250  nonzero_frac=0.1250
  genres_all   mean=0.0626  nonzero_frac=0.1250
  album        mean=0.0014  nonzero_frac=0.0014
  tags         mean=0.0004  nonzero_frac=0.0014

Logistic regression fit on 29972 sampled pairs:
  coef[genre] = 11.7992
  coef[genres_all] = 4.0748
  coef[album] = 0.0160
  coef[tags] = 0.0044
  intercept = -7.6334
```

Already an interesting, honest finding: the regression assigns `album`/`tags` almost no weight
(0.016 and 0.004 vs. genre's 11.8) — not a bug, a direct consequence of `album`/`tags` being so
sparse (nonzero for only 0.14% of pairs) that standard maximum-likelihood fitting can't extract
a confident signal from so few positive examples, even though those rare signals turn out to be
genuinely useful (see §6 below).

## 6. Evaluation — genre-cohesion@10

Same conventions as every other genre-cohesion figure in this project: k=10, sample_size=500,
seed=42 (`sonic_explorer.evaluation.genre_cohesion`, matching `scripts/run_evaluation.py`'s
real production config). `metadata_genre_cohesion_at_k` adapts the identical formula to run
directly over a dense similarity matrix instead of a FAISS index.

In [ ]:
from sonic_explorer.evaluation.genre_cohesion import metadata_genre_cohesion_at_k

K, SAMPLE_SIZE, SEED = 10, 500, 42

results = {}
for name, sims in variants.items():
    result = metadata_genre_cohesion_at_k(song_ids, sims, genre_by_song, k=K, sample_size=SAMPLE_SIZE, seed=SEED, label=name)
    results[name] = result
    print(f'{name:20s} observed={result.observed*100:5.1f}%  random_baseline={result.random_baseline*100:5.1f}%')

### Real result

```
equal_weight         observed=100.0%  random_baseline= 12.7%
genre_heavy          observed=100.0%  random_baseline= 12.7%
tags_album_heavy     observed= 99.8%  random_baseline= 12.7%
learned_logistic     observed=100.0%  random_baseline= 12.7%
```

**Honest finding: this metric saturates and can't differentiate these variants.** Every variant
that weights `genre_top` at all lands within noise of 100% — mechanistically, this is because
each genre has ~175 same-genre candidates in this library, vastly more than k=10, so *any*
nonzero genre weight guarantees the top-10 neighbor list fills with same-genre songs regardless
of the other three weights. This actually **quantitatively confirms** the tautology already
called out in the live app: a metadata baseline that includes genre is *structurally guaranteed*
near-perfect genre-cohesion, which is not the same thing as it having found real signal.

Since genre-cohesion@10 is saturated, it can't be the deciding metric here. The real
differentiator is how much genuine *cross-genre* connectivity each variant recovers — exactly
the `cross_genre_edge_fraction` metric the app already uses for this framing.

## 7. The real differentiator: cross-genre-edge fraction

Built on the app's actual `k_neighbors=4` default (via `_graph_from_similarity`, the same
k-NN + edge-selection logic `build_metadata_similarity_graph` uses), then
`cross_genre_edge_fraction` — already a real, tested function in `analysis/network_graph.py`
— measures what fraction of the resulting graph's edges connect two different genres. Since
genre-cohesion is tied, prefer the variant that recovers *more* genuine cross-genre signal, not
fewer: those crossings are real recovered metadata (shared album/tag), not noise.

In [ ]:
from sonic_explorer.analysis.network_graph import _graph_from_similarity, cross_genre_edge_fraction

cross_genre_pcts = {}
for name, sims in variants.items():
    graph_result = _graph_from_similarity(
        song_ids, sims, np.zeros((len(song_ids), 1)), k_neighbors=4, n_clusters=8, random_state=SEED
    )
    pct = cross_genre_edge_fraction(graph_result.edges, genre_by_song)
    cross_genre_pcts[name] = pct
    print(f'{name:20s} cross_genre_edge_fraction={pct*100:5.2f}%  (n_edges={len(graph_result.edges)})')

winner = max(cross_genre_pcts, key=cross_genre_pcts.get)
print(f'\nWinner: {winner} ({cross_genre_pcts[winner]*100:.2f}% cross-genre edges, '
      f'{results[winner].observed*100:.1f}% genre-cohesion@10)')

### Real result

```
equal_weight         cross_genre_edge_fraction= 0.02%  (n_edges=4354)
genre_heavy          cross_genre_edge_fraction= 0.00%  (n_edges=4353)
tags_album_heavy     cross_genre_edge_fraction= 0.39%  (n_edges=4361)
learned_logistic     cross_genre_edge_fraction= 0.00%  (n_edges=4354)

Winner: tags_album_heavy (0.39% cross-genre edges, 99.8% genre-cohesion@10)
```

## 8. Conclusion

**Winner: tags/album-heavy** (`genre=1, genres_all=1, album=2, tags=2`). Statistically tied with
every other variant on genre-cohesion@10 (99.8% vs. 100.0% — within noise at n=500 queries), but
produces **~20x more genuine cross-genre edges** than equal weighting (0.39% vs. 0.02%) at no
cost to the metric that matters. Up-weighting the rarer-but-more-specific album/tag signals lets
them occasionally outweigh a genre mismatch, surfacing real cross-genre connections a genre-only
(or equal-weighted) rule would miss.

The learned logistic-regression variant did **not** win — and the reason is itself a real,
explicable finding rather than a mystery: `album`/`tags` are too sparse (~0.14% of pairs have any
signal) for standard maximum-likelihood fitting to assign them meaningful weight against genre's
much denser signal, so it collapsed to an almost-genre-only model (cross_genre_edge_fraction
≈ 0%, same as genre-heavy). A more sophisticated learned approach (class-weighting, a prior
favoring the rare features, or waiting for real calibration-rating labels instead of using
genre-match as a proxy) might do better — a legitimate follow-up, not attempted here given this
was scoped as a basic, non-exhaustive attempt.

**This is now what ships:** `sonic_explorer/analysis/network_graph.py`'s `DEFAULT_METADATA_WEIGHTS`
was updated to `{'genre': 1.0, 'genres_all': 1.0, 'album': 2.0, 'tags': 2.0}` directly from this
result — the naive baseline shown in Overview §2 and Results §1 is the strongest defensible
version actually found, not whichever combination was quickest to implement first.